In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional

from src.utils import *
from src.analysis import *
from src.model import FineTunedModel



In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = FineTunedModel(num_classes=10).to(device)
model.load_state_dict(torch.load('weights/finetune_weights.pth', map_location=device))
target_layer = model.feature_extractor[5]
D = torch.load('weights/hals_nnd_D.pth')


In [ ]:
subset_paths = load_subset(subset_root="data/subset")
run_complete_analysis(
    model=model,
    target_layer=target_layer,
    D=torch.load('weights/hals_nnd_D.pth'),
    subset_paths=subset_paths,
    k=5,
    lam=1e-2
)

Loading subset from: data/subset
  cassette_player:  10 images (label=0)
  chain_saw   :  10 images (label=1)
  church      :  10 images (label=2)
  English_springer:  10 images (label=3)
  French_horn :  10 images (label=4)
  garbage_truck:  10 images (label=5)
  gas_pump    :  10 images (label=6)
  golf_ball   :  10 images (label=7)
  parachute   :  10 images (label=8)
  tench       :  10 images (label=9)

Total: 100 images loaded
Using device: cuda

Loading model and dictionary...

Analyzing 100 images...
Processing image 1/100...


/home/tamnt/.conda/envs/xfrt128/lib/python3.10/site-packages/torch/autograd/graph.py:841: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:270.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


In [ ]:
# ===== Analyze a single test image =====
print("\n" + "="*70)
print("Analyzing Test Image")
print("="*70)

# Option 1: Load image from path
image_path = "data/subset/church/1.jpeg"  # Change this to your image path
test_image = load_image(image_path)

# Option 2: Or use from dataloader (for testing with CIFAR-10)
# test_image, test_label = next(iter(data_loader))

# Create analyzer
analyzer = TopKActivationAnalyzer(model, target_layer, D, device)

# Get model prediction
with torch.no_grad():
    logits = model(test_image.to(device))
    pred_class = logits.argmax(dim=1).item()
    confidence = torch.softmax(logits, dim=1)[0, pred_class].item()

# class_names = get_cifar10_class_names()
print(f"Image path: {image_path}")
# print(f"Predicted class: {pred_class} ({class_names[pred_class]})")
print(f"Confidence: {confidence:.4f}")

# Analyze top-5 most influential activation maps
results = analyzer.visualize_analysis(
    image=test_image,
    k=5,
    class_idx=None,  # Use predicted class
    lam=1e-2
)

# Access results
print(f"Number of top maps analyzed: {len(results['top_maps'])}")
print(f"Top channel indices: {results['top_indices']}")


Analyzing Test Image
Image path: data/subset/church/1.jpeg
Confidence: 1.0000


/home/tamnt/.conda/envs/xfrt128/lib/python3.10/site-packages/torch/autograd/graph.py:841: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:270.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass



Top-5 Most Influential Activation Maps Analysis

Rank 1: Channel 133
  GradCAM Weight: 0.003079
  Sparse Code Stats:
    - Non-zero atoms: 23/500
    - L1 norm: 128.924126
    - Max coefficient: 33.415009
    - Top 5 atom indices: [458, 488, 255, 3, 99]
    - Top 5 coefficients: [33.41500879 13.54396237 10.99850564  9.87031173  9.30938522]

Rank 2: Channel 171
  GradCAM Weight: 0.002987
  Sparse Code Stats:
    - Non-zero atoms: 40/500
    - L1 norm: 107.072483
    - Max coefficient: 12.631390
    - Top 5 atom indices: [46, 410, 42, 458, 317]
    - Top 5 coefficients: [12.6313901   9.11219571  7.58708116  7.220824    6.5919924 ]

Rank 3: Channel 70
  GradCAM Weight: 0.002941
  Sparse Code Stats:
    - Non-zero atoms: 35/500
    - L1 norm: 48.954595
    - Max coefficient: 4.991638
    - Top 5 atom indices: [163, 408, 117, 9, 72]
    - Top 5 coefficients: [4.99163824 3.8602451  3.56657815 3.41254555 3.36344515]

Rank 4: Channel 40
  GradCAM Weight: 0.002903
  Sparse Code Stats:
    - No

In [ ]:
# ===== Analyze a single test image =====
print("\n" + "="*70)
print("Analyzing Test Image")
print("="*70)

# Option 1: Load image from path
image_path = "data/subset/church/2.jpeg"  # Change this to your image path
test_image = load_image(image_path)

# Option 2: Or use from dataloader (for testing with CIFAR-10)
# test_image, test_label = next(iter(data_loader))

# Create analyzer
analyzer = TopKActivationAnalyzer(model, target_layer, D, device)

# Get model prediction
with torch.no_grad():
    logits = model(test_image.to(device))
    pred_class = logits.argmax(dim=1).item()
    confidence = torch.softmax(logits, dim=1)[0, pred_class].item()

# class_names = get_cifar10_class_names()
print(f"Image path: {image_path}")
# print(f"Predicted class: {pred_class} ({class_names[pred_class]})")
print(f"Confidence: {confidence:.4f}")

# Analyze top-5 most influential activation maps
results = analyzer.visualize_analysis(
    image=test_image,
    k=5,
    class_idx=None,  # Use predicted class
    lam=1e-2
)

# Access results
print(f"Number of top maps analyzed: {len(results['top_maps'])}")
print(f"Top channel indices: {results['top_indices']}")


Analyzing Test Image
Image path: data/subset/church/2.jpeg
Confidence: 1.0000



Top-5 Most Influential Activation Maps Analysis

Rank 1: Channel 107
  GradCAM Weight: 0.002622
  Sparse Code Stats:
    - Non-zero atoms: 37/500
    - L1 norm: 117.536747
    - Max coefficient: 15.284773
    - Top 5 atom indices: [398, 358, 69, 367, 488]
    - Top 5 coefficients: [15.28477259 10.72983397  9.80085233  7.77807726  6.3592573 ]

Rank 2: Channel 158
  GradCAM Weight: 0.001978
  Sparse Code Stats:
    - Non-zero atoms: 52/500
    - L1 norm: 150.654694
    - Max coefficient: 16.621200
    - Top 5 atom indices: [220, 107, 163, 150, 267]
    - Top 5 coefficients: [16.62119967 12.92509668  8.99095405  6.27805355  5.99289525]

Rank 3: Channel 49
  GradCAM Weight: 0.001864
  Sparse Code Stats:
    - Non-zero atoms: 34/500
    - L1 norm: 181.055358
    - Max coefficient: 24.991443
    - Top 5 atom indices: [481, 121, 101, 148, 275]
    - Top 5 coefficients: [24.99144323 14.64298001 14.14822719 13.39991363 11.49059576]

Rank 4: Channel 122
  GradCAM Weight: 0.001735
  Sparse Code 

In [ ]:
# ===== Analyze a single test image =====
print("\n" + "="*70)
print("Analyzing Test Image")
print("="*70)

# Option 1: Load image from path
image_path = "data/subset/tench/2.jpeg"  # Change this to your image path
test_image = load_image(image_path)

# Option 2: Or use from dataloader (for testing with CIFAR-10)
# test_image, test_label = next(iter(data_loader))

# Create analyzer
analyzer = TopKActivationAnalyzer(model, target_layer, D, device)

# Get model prediction
with torch.no_grad():
    logits = model(test_image.to(device))
    pred_class = logits.argmax(dim=1).item()
    confidence = torch.softmax(logits, dim=1)[0, pred_class].item()

# class_names = get_cifar10_class_names()
print(f"Image path: {image_path}")
# print(f"Predicted class: {pred_class} ({class_names[pred_class]})")
print(f"Confidence: {confidence:.4f}")

# Analyze top-5 most influential activation maps
results = analyzer.visualize_analysis(
    image=test_image,
    k=5,
    class_idx=None,  # Use predicted class
    lam=1e-2
)

# Access results
print(f"Number of top maps analyzed: {len(results['top_maps'])}")
print(f"Top channel indices: {results['top_indices']}")


Analyzing Test Image
Image path: data/subset/tench/2.jpeg
Confidence: 1.0000



Top-5 Most Influential Activation Maps Analysis

Rank 1: Channel 70
  GradCAM Weight: 0.003473
  Sparse Code Stats:
    - Non-zero atoms: 47/500
    - L1 norm: 75.920263
    - Max coefficient: 8.724077
    - Top 5 atom indices: [480, 266, 487, 276, 290]
    - Top 5 coefficients: [8.72407666 5.99399445 5.08009459 4.06625707 3.52971572]

Rank 2: Channel 180
  GradCAM Weight: 0.002834
  Sparse Code Stats:
    - Non-zero atoms: 38/500
    - L1 norm: 185.273780
    - Max coefficient: 20.331029
    - Top 5 atom indices: [121, 46, 461, 287, 469]
    - Top 5 coefficients: [20.33102936 12.22601373 10.6719187  10.18448475  9.91432887]

Rank 3: Channel 136
  GradCAM Weight: 0.002716
  Sparse Code Stats:
    - Non-zero atoms: 51/500
    - L1 norm: 98.496717
    - Max coefficient: 8.989265
    - Top 5 atom indices: [136, 400, 181, 283, 275]
    - Top 5 coefficients: [8.98926519 6.43497246 4.28336776 4.27162288 4.12753213]

Rank 4: Channel 90
  GradCAM Weight: 0.002429
  Sparse Code Stats:
    - No